# [LAB-05] 1. 연습문제 - 보험료 가격 결정 구조 데이터셋 품질 검사

## #01. 준비작업

### 1. 필요한 라이브러리 참조

In [1]:
import numpy as np
from jussam import load_data
from pandas import DataFrame, read_excel

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 데이터 불러오기

In [2]:
origin = load_data("insurance")
origin.head()

📚 개인의 나이·성별·BMI·흡연 여부·거주 지역 등 기본 건강·인구학   적 정보를 바탕으로 의료보험 청구 비용(charges)을 예측하도록 구성된, 선형회귀와 머신러닝 실습에 널리 사용되는 대표적인 보험 비용 데이터셋 (출처: https://www.kaggle.com/datasets/mirichoi0218/insurance)

    field     description
--  --------  --------------------------------------------------------------
 0  age       가입자의 나이(세)
 1  sex       성별 (male, female)
 2  bmi       체질량 지수(Body Mass Index)
 3  children  부양 자녀 수(보험 내 자녀 수)
 4  smoker    흡연 여부 (yes / no)
 5  region    미국 내 거주 지역 (northeast, northwest, southeast, southwest)
 6  charges   의료보험 청구 비용(달러). 예측해야 하는 타깃 변수.



,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.924
1,18,male,33.770,1,no,southeast,1725.552
2,28,male,33.000,3,no,southeast,4449.462
3,33,male,22.705,0,no,northwest,21984.471
4,32,male,28.880,0,no,northwest,3866.855


## #02. 자료형 점검

### 1. 자료형 확인

In [3]:
origin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


### 2. 자료형 변환

In [4]:
# 원본 데이터 프레임 복사
df1 = origin.copy()
df1["sex"] = df1["sex"].astype("category")
df1["smoker"] = df1["smoker"].astype("category")
df1["region"] = df1["region"].astype("category")
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1338 non-null   int64   
 1   sex       1338 non-null   category
 2   bmi       1338 non-null   float64 
 3   children  1338 non-null   int64   
 4   smoker    1338 non-null   category
 5   region    1338 non-null   category
 6   charges   1338 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.3 KB


## #03. 데이터 중복 점검

### 1. 행단위 완전 중복 확인

In [5]:
dup = df1.duplicated()
dup

0       False
1       False
2       False
3       False
4       False
        ...  
1333    False
1334    False
1335    False
1336    False
1337    False
Length: 1338, dtype: bool

In [6]:
# 중복 데이터 개수
dup.sum()

np.int64(1)

> 앗! 중복 데이터가 발견되었다.

### 2. 중복 데이터 제거

In [8]:
df2 = df1.drop_duplicates()

# 중복 제거 후 중복 개수 다시 확인
df2.duplicated().sum()

np.int64(0)

## #04. 형식·표기·단위 점검

### 1. 명목형 변수의 단위 확인

In [9]:
fields = df2.select_dtypes(include=["category"]).columns

for field in fields:
    display(df2[field].value_counts())

sex
male      675
female    662
Name: count, dtype: int64

smoker
no     1063
yes     274
Name: count, dtype: int64

region
southeast    364
southwest    325
northeast    324
northwest    324
Name: count, dtype: int64

> 데이터의 종류에는 문제가 없으나, 라벨링 처리는 필요하다.

### 2. 연속형 변수의 단위 확인을 위한 변수명 추출

In [10]:
# 값을 검사 수치형 변수의 이름 추출
fields = df2.select_dtypes(include="number").columns.to_list()
print(fields)

['age', 'bmi', 'children', 'charges']


### 3. 연속형 변수의 단위 확인

In [11]:
minmax = [] # 최소값과 최대값을 저장할 리스트

for field in fields:
    min_value = df2[field].min()
    max_value = df2[field].max()
    minmax.append({"min": min_value, "max": max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력
minmax_df = DataFrame(minmax, index=fields)
minmax_df

,min,max
age,18.000,64.000
bmi,15.960,53.130
children,0.000,5.000
charges,1121.874,63770.428


> 데이터 명세의 내용에서 크게 벗어나는 범위는 없는것으로 판단됨

## #05. 결측치 점검

### 1. 결측치 확인

In [12]:
na_count = df2.isna().sum()
na_count

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

### 2. 결측치 비율 계산을 위한 데이터 크기 확인

In [13]:
# 데이터의 행과 열의 수 확인
rows, cols = df2.shape
print(f"rows: {rows}, cols: {cols}")

rows: 1337, cols: 7


### 3. 결측치 비율 확인

In [14]:
na_ratio = na_count / rows
na_ratio

age        0.000
sex        0.000
bmi        0.000
children   0.000
smoker     0.000
region     0.000
charges    0.000
dtype: float64

## #06. 품질 점검 결과 데이터셋 저장

In [ ]:
df2.to_excel("insurance_qtcheck.xlsx", index=False)

## #07. 기술 통계량 확인 준비작업

### 1. 데이터 불러오기

In [ ]:
#origin_qt = read_excel("insurance_qtcheck.xlsx")

# 수업용 자료이므로 라이브러리를 통해서도 로드 가능함
origin_qt = load_data("insurance_qtcheck")

### 2. 타입변환

- category 타입으로의 변환은 데이터를 불러올 때 마다 진행해야 한다.

In [18]:
df3 = origin_qt.copy()
df3["sex"] = df3["sex"].astype("category")
df3["smoker"] = df3["smoker"].astype("category")
df3["region"] = df3["region"].astype("category")
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   age       1337 non-null   int64   
 1   sex       1337 non-null   category
 2   bmi       1337 non-null   float64 
 3   children  1337 non-null   int64   
 4   smoker    1337 non-null   category
 5   region    1337 non-null   category
 6   charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 46.3 KB


## #08. 기술 통계량 표 생성

### 1. 연속형 변수의 기술 통계량 표

- 명목형 변수는 자동으로 제외된다.

In [19]:
# T는 데이터프레임의 행과 열을 바꿔주는 역할
desc_df = df3.describe().T
desc_df

,count,mean,std,min,25%,50%,75%,max
age,1337.000,39.222,14.044,18.000,27.000,39.000,51.000,64.000
bmi,1337.000,30.663,6.100,15.960,26.290,30.400,34.700,53.130
children,1337.000,1.096,1.206,0.000,0.000,1.000,2.000,5.000
charges,1337.000,13279.121,12110.360,1121.874,4746.344,9386.161,16657.717,63770.428


### 2. 명목형 변수의 기술 통계량 표

- 데이터의 수(count), 값의 종류 수(unique), 최빈값(top), 최빈값의 수(freq)를 알 수 있다.

In [20]:
cate_desc_df = df3.describe(include='category').T
cate_desc_df

,count,unique,top,freq
sex,1337,2,male,675
smoker,1337,2,no,1063
region,1337,4,southeast,364


### 3. 명목형 변수 기술 통계량

- 명목형 변수는 값의 종류별로 데이터 수와 비율을 확인할 수 있다.

In [21]:
# 명목형 변수의 필드명 추출
cate_fields = df3.select_dtypes(include='category').columns

# 명목형 변수의 고유값과 빈도수 계산
for field in cate_fields:
    # 값의 종류별 개수
    vcount = df3[field].value_counts()
    # 전체 데이터 길이 대비 비율
    percent = vcount / df3.shape[0]

    # 결과 저장
    df = DataFrame({'count': vcount, 'percent': percent})
    # 결과 출력
    display(df)

,count,percent
sex,,
male,675,0.505
female,662,0.495


,count,percent
smoker,,
no,1063,0.795
yes,274,0.205


,count,percent
region,,
southeast,364,0.272
southwest,325,0.243
northeast,324,0.242
northwest,324,0.242


## #09. 중심 수준 파악

### 1. **평균-중앙값의 상대 차이율** 계산

- 평균-중앙값 상대 차이율 = |평균 - 중앙값| / 중앙값

| 구간 | 의미 |
|--------|-----------------|
| 0 ~ 0.1   | 평균과 중앙값이 거의 비슷함 |
| 0.1 ~ 0.5  | 약간 차이 있음        |
| 0.5 | 차이 큼, 왜도·극단값 의심 |

In [22]:
# "평균-중앙값 상대 차이율 = |평균 - 중앙값| / 중앙값" 컬럼 추가
desc_df['rel_diff'] = abs(desc_df['mean'] - desc_df['50%']) / desc_df['50%']

# 상대 차이율 의미 컬럼 추가
conditions = [desc_df['rel_diff'] < 0.1, desc_df['rel_diff'] < 0.5]
choices = ['similar', 'diff']
desc_df['rdiff_flag'] = np.select(conditions, choices, default='large_diff')

# 필요한 컬럼만 선택하여 출력
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag']]

,mean,50%,rel_diff,rdiff_flag
age,39.222,39.000,0.006,similar
bmi,30.663,30.400,0.009,similar
children,1.096,1.000,0.096,similar
charges,13279.121,9386.161,0.415,diff


## #10. 이상치 신호 탐지

### 1. IQR, 이상치 경계값 계산

In [23]:
# iqr
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']

# 상한 이상치 경계
desc_df['upper_bound'] = desc_df['75%'] + 1.5 * desc_df['iqr']

# 하한 이상치 경계
desc_df['lower_bound'] = desc_df['25%'] - 1.5 * desc_df['iqr']

# 필요한 컬럼만 선택하여 출력
desc_df[['iqr', 'upper_bound', 'lower_bound']]

,iqr,upper_bound,lower_bound
age,24.000,87.000,-9.000
bmi,8.410,47.315,13.675
children,2.000,5.000,-3.000
charges,11911.373,34524.778,-13120.716


### 2. 명목형 변수를 제외한 데이터 프레임

- 이상치는 연속형 변수에 대한 개념이므로 명목형 변수를 제외한 데이터프레임 생성

In [24]:
# 명목형 변수의 필드명 추출
cate_fields = df3.select_dtypes(include='category').columns

df4 = df3.drop(columns=cate_fields)
df4.head()

,age,bmi,children,charges
0,19,27.900,0,16884.924
1,18,33.770,1,1725.552
2,28,33.000,3,4449.462
3,33,22.705,0,21984.471
4,32,28.880,0,3866.855


### 3. 상한 이상치 탐지

In [25]:
# 상한 이상치 수
desc_df['upper_outliers'] = ((df4 > desc_df['upper_bound'])).sum()

# 상한 이상치 수 비율
desc_df['upper_outliers_ratio'] = desc_df['upper_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['upper_outliers', 'upper_outliers_ratio']]

,upper_outliers,upper_outliers_ratio
age,0,0.000
bmi,9,0.007
children,0,0.000
charges,139,0.104


### 4. 하한 이상치 탐지

In [26]:
# 하한 이상치 수
desc_df['lower_outliers'] = (df4 < desc_df['lower_bound']).sum()

# 하한 이상치 수 비율
desc_df['lower_outliers_ratio'] = desc_df['lower_outliers'] / df4.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['lower_outliers', 'lower_outliers_ratio']]

,lower_outliers,lower_outliers_ratio
age,0,0.000
bmi,0,0.000
children,0,0.000
charges,0,0.000


### 5. 전체 이상치 집계

In [27]:
# 통합 이상치 수
desc_df['outliers'] = desc_df['upper_outliers'] + desc_df['lower_outliers']

# 통합 이상치 수 비율
desc_df['outliers_ratio'] = desc_df['outliers'] / df4.shape[0]

# 이상치 탐지 결과 확인
desc_df[['upper_outliers', 'upper_outliers_ratio', 
         'lower_outliers', 'lower_outliers_ratio', 
         'outliers', 'outliers_ratio']]

,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio
age,0,0.000,0,0.000,0,0.000
bmi,9,0.007,0,0.000,9,0.007
children,0,0.000,0,0.000,0,0.000
charges,139,0.104,0,0.000,139,0.104


## #11. 비대칭 신호 확인

### 1. 왜도 점검

In [28]:
# 왜도 계산
desc_df['skew'] = df4.skew()

# 왜도를 통한 분포 형태 해석
conditions_skew = [(desc_df['skew'] < -0.5), (desc_df['skew'] > 0.5)]
choices_skew = ['left tail', 'right tail']
desc_df['skew_interpret'] = np.select(conditions_skew, choices_skew,
                                    default='symmetric')

# 필요한 컬럼만 선택하여 출력
desc_df[['skew', 'skew_interpret']]

,skew,skew_interpret
age,0.055,symmetric
bmi,0.284,symmetric
children,0.937,right tail
charges,1.515,right tail


### 2. 첨도 점검

In [29]:
# 첨도 계산
desc_df['kurt'] = df4.kurt()

# 첨도를 통한 분포 형태 해석
conditions_kurt = [(desc_df['kurt'] < 0), (desc_df['kurt'] > 0)]
choices_kurt = ['platykurtic', 'leptokurtic']
desc_df['kurt_interpret'] = np.select(conditions_kurt, choices_kurt,
                                    default='mesokurtic')

# 필요한 컬럼만 선택하여 출력
desc_df[['kurt', 'kurt_interpret']]

,kurt,kurt_interpret
age,-1.244,platykurtic
bmi,-0.053,platykurtic
children,0.201,leptokurtic
charges,1.604,leptokurtic


### 3. 로그 변환 필요성 판단 함수 정의

In [30]:
def judge_log_transform(skew, kurt):
    if skew >= 1:                        # 강한 우측 꼬리 분포
        return "log1p"
    elif skew > 0.5 and kurt > 0:       # 우측 꼬리 분포이면서 첨도가 높은 경우
        return "log1p"
    elif skew <= -1:                    # 강한 좌측 꼬리 분포
        return "reverse_log1p"
    elif skew < -0.5 and kurt > 0:      # 좌측 꼬리 분포이면서 첨도가 높은 경우
        return "reverse_log1p"
    else:                               # 대칭 분포
        return "none"

### 4. 로그 변환 필요성 판정

In [31]:
desc_df['log_need'] = desc_df.apply(lambda row: judge_log_transform( row['skew'], row['kurt']), axis=1)

# 로그 변환 필요성 결과 확인
desc_df[['skew', 'kurt', 'log_need']]

,skew,kurt,log_need
age,0.055,-1.244,none
bmi,0.284,-0.053,none
children,0.937,0.201,log1p
charges,1.515,1.604,log1p


## #12. 최종 기술 통계량 확인

### 1. 최종 통계량 표 확인

In [32]:
desc_df.T

,age,bmi,children,charges
count,1337.000,1337.000,1337.000,1337.000
mean,39.222,30.663,1.096,13279.121
std,14.044,6.100,1.206,12110.360
min,18.000,15.960,0.000,1121.874
25%,27.000,26.290,0.000,4746.344
50%,39.000,30.400,1.000,9386.161
75%,51.000,34.700,2.000,16657.717
max,64.000,53.130,5.000,63770.428
rel_diff,0.006,0.009,0.096,0.415
rdiff_flag,similar,similar,similar,diff


### 2. 기술 통계량 표 저장

In [ ]:
desc_df.to_excel("insurance_qtcheck_desc.xlsx", index=True)

## 💡 인사이트

**1. Charges (의료보험 청구 비용) - 주의 필요**
- 평균(13,279)과 중앙값(9,386) 차이가 큼 (상대차이율 0.415) → 비대칭 분포
- 왜도 1.515로 **강한 우측 꼬리 분포** → 고액 의료비가 우측에 집중
- 상한 이상치 139개(10.4%) → 상당한 극값 존재
- **로그 변환(log1p) 필수** 권장

**2. Children (부양 자녀 수)**
- 왜도 0.937로 우측 꼬리 분포
- 이상치는 없으나 **로그 변환 권장** (분포 정상화)

**3. Age, BMI - 건강함**
- 평균-중앙값 거의 일치 (rel_diff < 0.01)
- 대칭적 분포, 이상치 거의 없음
- 추가 처리 불필요

**4. 데이터 품질**
- 결측치 0건, 완전중복 1건 제거 완료 (1,338→1,337)
- 전체적으로 양호한 상태